In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import xgboost as xgb
import joblib

print("done")


The below code snippet outlines a process for financial data analysis and predictive modeling using Python. Here's a step-by-step explanation:

### Step 1: Information Gathering
The `get_data` function downloads historical stock data (adjusted closing prices) for a list of specified tickers (e.g., AAPL, MSFT) from Yahoo Finance using `yf.download`. It handles exceptions and returns a Pandas DataFrame containing the adjusted closing prices.

### Step 2: Feature Engineering
The `feature_engineering` function computes two features per ticker: daily returns (`{ticker}_return`) and volatility (`{ticker}_volatility`) over a rolling 21-day window. It drops rows with missing values (NaNs) and returns a cleaned DataFrame.

### Data Splitting
The data is split into three periods:
- `train_period` (2010-2014): Training data used to fit machine learning models.
- `validation_period` (2015-2019): Validation data used to tune model parameters and prevent overfitting.
- `test_period` (2020-2022): Test data used to evaluate the model's performance on unseen data.

### Step 3: Machine Learning Models and Saving
The `train_and_save_models` function trains XGBoost regressor models to predict future stock returns for each ticker based on the engineered features. It performs the following:
- Scales features using `StandardScaler`.
- Reduces dimensionality using `PCA` to retain 95% variance.
- Trains an `XGBRegressor` model for each ticker's return using principal components from PCA.
- Saves trained models (`joblib`) and scaling/PCA objects for future use.

### Example Usage
The script demonstrates loading a saved model (`joblib.load`) and making predictions (`predict`) on unseen test data (`X_test_pca`).

Finally, it prints "Models saved successfully." upon completion.

### Additional Note
The `prepare test data` section at the end of the code snippet prepares the test data (`X_test`) by scaling it using the previously fitted `StandardScaler` and transforming it using the saved `PCA` model (`X_test_pca`).



In [ ]:
# Step 1: Information Gathering
def get_data(tickers, start_date, end_date):
    data = {}
    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start_date, end=end_date)
            data[ticker] = df['Adj Close']
        except Exception as e:
            print(f"Error downloading {ticker}: {e}")
    return pd.DataFrame(data)

tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "META"]  # Updated to 'META' as 'FB' is delisted
start_date = '2010-01-01'
end_date = '2022-12-31'
data = get_data(tickers, start_date, end_date)

# Step 2: Feature Engineering
def feature_engineering(data):
    df = pd.DataFrame()
    for ticker in tickers:
        df[f'{ticker}_return'] = data[ticker].pct_change()
        df[f'{ticker}_volatility'] = df[f'{ticker}_return'].rolling(window=21).std()
    df = df.dropna()
    return df

features = feature_engineering(data)

# Split data into training, validation, and test sets
train_period = features['2010':'2014']
validation_period = features['2015':'2019']
test_period = features['2020':'2022']

# Step 3: Machine Learning Models and Saving
def train_and_save_models(train_period, validation_period, save_path="models/"):
    # Create the directory if it doesn't exist
    os.makedirs(save_path, exist_ok=True)

    X_train = train_period.drop(columns=[col for col in train_period.columns if 'return' in col])
    y_train = train_period[[col for col in train_period.columns if 'return' in col]]

    X_val = validation_period.drop(columns=[col for col in validation_period.columns if 'return' in col])
    y_val = validation_period[[col for col in validation_period.columns if 'return' in col]]

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Dimensionality Reduction
    pca = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    # Model training
    models = {}
    for col in y_train.columns:
        model = xgb.XGBRegressor(objective='reg:squarederror')
        model.fit(X_train_pca, y_train[col], eval_set=[(X_val_pca, y_val[col])], early_stopping_rounds=50, verbose=False)

        # Save the trained model using joblib
        joblib.dump(model, f"{save_path}{col}_model.joblib")
        models[col] = model

    # Save the scaler and pca objects
    joblib.dump(scaler, f"{save_path}scaler.joblib")
    joblib.dump(pca, f"{save_path}pca.joblib")

    return models, scaler, pca

models, scaler, pca = train_and_save_models(train_period, validation_period)

# Example of how to load the model and use it
# loaded_model = joblib.load("models/AAPL_return_model.joblib")
# pred = loaded_model.predict(X_test_pca)

print("Models saved successfully.")


# Prepare test data
X_test = test_period.drop(columns=[col for col in test_period.columns if 'return' in col])
y_test = test_period[[col for col in test_period.columns if 'return' in col]]
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)# Step 1: Information Gathering
def get_data(tickers, start_date, end_date):
    data = {}
    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start_date, end=end_date)
            data[ticker] = df['Adj Close']
        except Exception as e:
            print(f"Error downloading {ticker}: {e}")
    return pd.DataFrame(data)

tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "META"]  # Updated to 'META' as 'FB' is delisted
start_date = '2010-01-01'
end_date = '2022-12-31'
data = get_data(tickers, start_date, end_date)

# Step 2: Feature Engineering
def feature_engineering(data):
    df = pd.DataFrame()
    for ticker in tickers:
        df[f'{ticker}_return'] = data[ticker].pct_change()
        df[f'{ticker}_volatility'] = df[f'{ticker}_return'].rolling(window=21).std()
    df = df.dropna()
    return df

features = feature_engineering(data)

# Split data into training, validation, and test sets
train_period = features['2010':'2014']
validation_period = features['2015':'2019']
test_period = features['2020':'2022']

# Step 3: Machine Learning Models and Saving
def train_and_save_models(train_period, validation_period, save_path="models/"):
    # Create the directory if it doesn't exist
    os.makedirs(save_path, exist_ok=True)

    X_train = train_period.drop(columns=[col for col in train_period.columns if 'return' in col])
    y_train = train_period[[col for col in train_period.columns if 'return' in col]]

    X_val = validation_period.drop(columns=[col for col in validation_period.columns if 'return' in col])
    y_val = validation_period[[col for col in validation_period.columns if 'return' in col]]

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Dimensionality Reduction
    pca = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    # Model training
    models = {}
    for col in y_train.columns:
        model = xgb.XGBRegressor(objective='reg:squarederror')
        model.fit(X_train_pca, y_train[col], eval_set=[(X_val_pca, y_val[col])], early_stopping_rounds=50, verbose=False)

        # Save the trained model using joblib
        joblib.dump(model, f"{save_path}{col}_model.joblib")
        models[col] = model

    # Save the scaler and pca objects
    joblib.dump(scaler, f"{save_path}scaler.joblib")
    joblib.dump(pca, f"{save_path}pca.joblib")

    return models, scaler, pca

models, scaler, pca = train_and_save_models(train_period, validation_period)

# Example of how to load the model and use it
# loaded_model = joblib.load("models/AAPL_return_model.joblib")
# pred = loaded_model.predict(X_test_pca)

print("Models saved successfully.")


# Prepare test data
X_test = test_period.drop(columns=[col for col in test_period.columns if 'return' in col])
y_test = test_period[[col for col in test_period.columns if 'return' in col]]
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

print("done")

### Step 4: Mean-Variance Optimization using `scipy.optimize`

This section focuses on optimizing the portfolio allocation based on mean-variance analysis using the Sharpe ratio as the objective function. Here?s an explanation of the code:

#### `objective` Function
The `objective` function is defined to calculate the negative Sharpe ratio, which will be minimized by the optimizer (`scipy.optimize`). The function takes the following parameters:
- `weights`: A vector representing the allocation of assets in the portfolio.
- `expected_returns`: Expected returns of assets in the portfolio.
- `cov_matrix`: Covariance matrix representing the risk (volatility) and correlation between assets.
- `risk_free_rate`: The risk-free rate, used in the Sharpe ratio calculation.

Inside the function:
1. **Portfolio Return Calculation**: `portfolio_return` computes the expected return of the portfolio based on the weighted sum of asset returns (`weights` and `expected_returns`).
   
2. **Portfolio Volatility Calculation**: `portfolio_volatility` computes the portfolio volatility (standard deviation) using the formula:
   \[
   \text{portfolio\_volatility} = \sqrt{\text{weights}^\top \cdot \text{cov\_matrix} \cdot \text{weights}}
   \]
   This calculates the risk of the portfolio based on the covariance matrix.

3. **Sharpe Ratio Calculation**: `sharpe_ratio` computes the Sharpe ratio, which measures the risk-adjusted return of the portfolio relative to the risk-free rate:
   \[
   \text{sharpe\_ratio} = \frac{\text{portfolio\_return} - \text{risk\_free\_rate}}{\text{portfolio\_volatility}}
   \]
   A higher Sharpe ratio indicates better risk-adjusted performance.

4. **Return Value**: The function returns `-sharpe_ratio` because `scipy.optimize` minimizes functions by default, and we want to maximize the Sharpe ratio.

#### `print("done")`
Finally, the script prints "done" to indicate the completion of this section. This typically signifies the end of the optimization process or the execution of the optimization function.


In [ ]:
# Step 4: Mean-Variance Optimization using scipy.optimize
def objective(weights, expected_returns, cov_matrix, risk_free_rate):
    portfolio_return = np.dot(weights, expected_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return -sharpe_ratio

print("done")

### Optimization of Portfolio Allocation using `scipy.optimize`

This code segment demonstrates the optimization of portfolio allocation based on expected returns and covariance matrix, aiming to maximize the Sharpe ratio. Here?s a breakdown of the process:

#### Import Statement
The `minimize` function from `scipy.optimize` is imported to facilitate the optimization process.

#### `optimize_portfolio` Function
The `optimize_portfolio` function is defined to find the optimal portfolio allocation. Here?s a step-by-step explanation:

1. **Initialization**:
   - `n` is the number of assets (length of `expected_returns`).
   - `initial_weights` initializes the portfolio with equal weights (`1 / n` for each asset).
   - `bounds` sets constraints on the optimization variables (`0` to `1` for each weight, indicating they must sum to `1`).
   
2. **Constraints**:
   - `constraints` specifies that the sum of weights must equal `1` (fully invested).

3. **Minimization**:
   - `minimize` optimizes the objective function (`objective`) with initial weights (`initial_weights`), and additional arguments (`expected_returns`, `cov_matrix`, `risk_free_rate`).
   - `bounds` and `constraints` ensure that weights are non-negative and sum to `1`.

4. **Result**:
   - `result.x` contains the optimal weights found by the optimizer.
   - `optimal_portfolio_return` calculates the expected return of the optimal portfolio.
   - `optimal_portfolio_volatility` calculates the volatility (risk) of the optimal portfolio.
   - `optimal_sharpe_ratio` computes the Sharpe ratio of the optimal portfolio, measuring its risk-adjusted return.

5. **Return**:
   - The function returns `optimal_weights` (allocation), and `optimal_sharpe_ratio` (maximized Sharpe ratio).

#### Example Usage
After defining the function, the script calculates:
- `predicted_returns`: Predicted returns from the models (assuming it's a 2D array where each row corresponds to predictions for different assets).
- `cov_matrix`: Covariance matrix of predicted returns to estimate risk.
- `expected_returns`: Mean of predicted returns across assets.
- `risk_free_rate`: Assumed risk-free rate (here set to `0` for simplicity).

Then, it calls `optimize_portfolio` with `expected_returns`, `cov_matrix`, and `risk_free_rate` to find optimal weights and maximize the Sharpe ratio.

#### Printing Results
Finally, the script prints:
- `Optimal Portfolio Weights`: The optimal allocation of assets.
- `Max Sharpe Ratio`: The maximum Sharpe ratio achieved by the portfolio.

This process helps investors optimize their portfolio allocation based on predicted returns and risk considerations.



In [ ]:
from scipy.optimize import minimize
def optimize_portfolio(expected_returns, cov_matrix, risk_free_rate):
    n = len(expected_returns)
    initial_weights = np.array([1 / n] * n)  # Initialize with equal weights
    bounds = [(0, 1)] * n  # Bounds for each weight

    # Constraints: sum of weights equals 1
    constraints = ({'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1})

    result = minimize(objective, initial_weights, args=(expected_returns, cov_matrix, risk_free_rate),
                      bounds=bounds, constraints=constraints)

    optimal_weights = result.x
    optimal_portfolio_return = np.dot(optimal_weights, expected_returns)
    optimal_portfolio_volatility = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))
    optimal_sharpe_ratio = (optimal_portfolio_return - risk_free_rate) / optimal_portfolio_volatility

    return optimal_weights, optimal_sharpe_ratio

predicted_returns = np.array([model.predict(X_test_pca) for model in models.values()]).T
cov_matrix = np.cov(predicted_returns, rowvar=False)

# Example usage
expected_returns = predicted_returns.mean(axis=0)
risk_free_rate = 0.0  # Assuming risk-free rate is zero for simplicity
optimal_weights, max_sharpe_ratio = optimize_portfolio(expected_returns, cov_matrix, risk_free_rate)

print(f'Optimal Portfolio Weights: {optimal_weights}')
print(f'Max Sharpe Ratio: {max_sharpe_ratio}')

### Step 5: Backtesting

The code snippet outlines the process of backtesting a portfolio based on historical data and optimal portfolio weights obtained from the optimization step. Here?s a detailed explanation:

#### `backtest_portfolio` Function

The `backtest_portfolio` function is designed to simulate the performance of the optimized portfolio over historical data. Here?s how it works:

1. **Calculate Daily Returns**:
   - `daily_returns = data.pct_change().dropna()` computes the daily returns of the asset prices (or portfolio value) from the provided historical data. It drops the first row (NaN) resulting from the percentage change calculation.

2. **Portfolio Return Calculation**:
   - `portfolio_return = daily_returns.dot(optimal_weights)` calculates the daily portfolio returns by taking the dot product of `daily_returns` (a DataFrame of daily returns) and `optimal_weights` (the optimized portfolio allocation obtained previously).

3. **Cumulative Return Calculation**:
   - `(1 + portfolio_return).cumprod() - 1` computes the cumulative return of the portfolio over time. It adds 1 to `portfolio_return` (to get cumulative growth factor), computes the cumulative product (to get cumulative portfolio value), and subtracts 1 to express the cumulative return as a percentage.

4. **Return**:
   - The function returns `cumulative_return`, which represents the cumulative performance of the optimized portfolio over the backtesting period.

#### Example Usage

After defining the `backtest_portfolio` function, the script executes the following:

- `cumulative_return = backtest_portfolio(data['2020':], optimal_weights)` calls the function to backtest the portfolio using historical data from the year 2020 onward (`data['2020':]`) and the `optimal_weights` obtained from the optimization step.

- `print(f'Cumulative Return: {cumulative_return[-1]}')` prints the final cumulative return of the portfolio. `cumulative_return[-1]` accesses the last value in the cumulative return series, indicating the overall return achieved by the optimized portfolio by the end of the backtesting period.

### Summary

This step allows for evaluating how well the optimized portfolio allocation would have performed historically, providing insights into its potential performance in real-world scenarios based on past market behavior.


In [ ]:
# Step 5: Backtesting
def backtest_portfolio(data, optimal_weights):
    daily_returns = data.pct_change().dropna()
    portfolio_return = daily_returns.dot(optimal_weights)
    cumulative_return = (1 + portfolio_return).cumprod() - 1

    return cumulative_return

cumulative_return = backtest_portfolio(data['2020':], optimal_weights)
print(f'Cumulative Return: {cumulative_return[-1]}')

### Visualization

The provided code segment aims to visually represent the cumulative return of the optimal portfolio over time using a matplotlib plot. Here?s an explanation of each component:

#### `plt.plot(cumulative_return)`
- `plt.plot()` is a matplotlib function used to create a line plot.
- `cumulative_return` is the data array representing the cumulative returns of the optimal portfolio over different time periods.

#### `plt.title('Cumulative Return of Optimal Portfolio')`
- `plt.title()` sets the title of the plot to 'Cumulative Return of Optimal Portfolio'. This title provides context about what the plot illustrates.

#### `plt.xlabel('Date')`
- `plt.xlabel()` labels the x-axis of the plot with 'Date'. This axis represents the timeline or dates corresponding to the cumulative returns.

#### `plt.ylabel('Cumulative Return')`
- `plt.ylabel()` labels the y-axis of the plot with 'Cumulative Return'. This axis represents the cumulative return values of the portfolio.

#### `plt.show()`
- `plt.show()` displays the plot. In a Jupyter Notebook or similar environment, this function call renders the plot directly below the code cell.

### Summary

This visualization helps stakeholders quickly grasp how the optimal portfolio has performed over time. It visually represents the cumulative growth or decline of the portfolio's value, based on historical data and the optimized allocation strategy derived earlier. Such visualizations are crucial for performance analysis and decision-making in investment scenarios.


In [ ]:
# Visualization
plt.plot(cumulative_return)
plt.title('Cumulative Return of Optimal Portfolio')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.show()

In [ ]:
pip install cvxopt

## Below is A function to Test Trained Model


 This is only used for testing purposes.

### Testing the Trained Model

This section demonstrates how to test a previously trained model using simulated test data. It involves generating sample data, performing feature engineering, training a simple model, optimizing the portfolio, and evaluating its performance. Here?s a step-by-step explanation:

#### Import Libraries
The necessary libraries for data manipulation, model training, and optimization are imported:
- `pandas` for handling data.
- `numpy` for numerical operations.
- `DummyRegressor` from `sklearn.dummy` for creating a simple model.
- `joblib` for saving and loading models.
- `matrix` and `solvers` from `cvxopt` for quadratic programming.
- `matplotlib.pyplot` for plotting.

#### Generate Sample Test Data
Sample test data is generated to simulate stock prices and volumes:
- `dates` creates a date range from January 1, 2023, to January 10, 2023.
- `data` dictionary contains simulated price and volume data for AAPL and MSFT.
- `test_data` DataFrame is created from the `data` dictionary and indexed by `dates`.
- Moving averages (20-day) for AAPL and MSFT prices are calculated and added to `test_data`.

#### Step 1: Information Gathering
The `get_data` function is defined to retrieve data (simulated in this example):
- `tickers`, `start_date`, and `end_date` are parameters.
- The function returns `test_data`.

#### Step 2: Feature Engineering
The `feature_engineering` function processes the data to create features:
- Calculates daily returns and volatility (5-day rolling standard deviation) for AAPL and MSFT.
- Returns a DataFrame with these features, dropping any NaN values.

#### Step 3: Machine Learning Models
The `train_and_save_models` function trains and saves simple models:
- Uses `DummyRegressor` as a placeholder model, which predicts the mean of the target variable.
- Trains models on the training period and saves them using `joblib`.

#### Step 4: Mean-Variance Optimization
The `mean_variance_optimization` function performs portfolio optimization:
- Uses quadratic programming to solve for optimal portfolios that maximize returns for different risk levels.
- Computes efficient frontier portfolios, their returns, and risks.

#### Step 5: Portfolio Evaluation
The `evaluate_portfolio` function assesses the portfolios:
- Calculates Sharpe ratios for the portfolios.
- Identifies the portfolio with the maximum Sharpe ratio (optimal portfolio).

#### Step 6: Example Usage with Sample Test Data
Example usage demonstrates the full workflow:
- Retrieves data using `get_data`.
- Performs feature engineering.
- Splits data into training, validation, and test periods.
- Trains models on the training period.
- Predicts returns using the trained models on the test period.
- Calculates the covariance matrix of predicted returns.
- Performs mean-variance optimization to find optimal portfolios.
- Evaluates the optimal portfolio and its Sharpe ratio.
- Plots the efficient frontier and highlights the optimal portfolio.

This example workflow provides a comprehensive demonstration of how to test and evaluate a machine learning model for financial portfolio optimization using simulated data.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyRegressor
import joblib
from cvxopt import matrix, solvers
import matplotlib.pyplot as plt

# Generate sample test data
dates = pd.date_range(start='2023-01-01', end='2023-01-10')
np.random.seed(0)

data = {
    'AAPL_Price': np.random.normal(loc=150, scale=5, size=len(dates)),
    'MSFT_Price': np.random.normal(loc=300, scale=10, size=len(dates)),
    'AAPL_Volume': np.random.randint(1000000, 2000000, size=len(dates)),
    'MSFT_Volume': np.random.randint(500000, 1000000, size=len(dates))
}

test_data = pd.DataFrame(data, index=dates)
test_data['AAPL_MA_20'] = test_data['AAPL_Price'].rolling(window=20).mean()  # Example moving average
test_data['MSFT_MA_20'] = test_data['MSFT_Price'].rolling(window=20).mean()  # Example moving average

print("Sample Test Data:")
print(test_data)

# Step 1: Information Gathering
def get_data(tickers, start_date, end_date):
    # Simulated data; replace with actual data retrieval logic if needed
    return test_data

# Step 2: Feature Engineering (simplified for demonstration)
def feature_engineering(data):
    # Simulated feature engineering; replace with actual feature engineering logic
    df = pd.DataFrame()
    for ticker in ['AAPL', 'MSFT']:  # Assuming we know the tickers in advance
        df[f'{ticker}_return'] = data[f'{ticker}_Price'].pct_change()
        df[f'{ticker}_volatility'] = df[f'{ticker}_return'].rolling(window=5).std()  # Example volatility
    df = df.dropna()
    return df

# Step 3: Machine Learning Models
def train_and_save_models(train_period, validation_period):
    # Simplified training and saving of models
    models = {}
    for ticker in ['AAPL', 'MSFT']:  # Assuming we know the tickers in advance
        model = DummyRegressor(strategy='mean')  # Example model, replace with your actual model
        model.fit(train_period.drop(columns=[f'{ticker}_return']), train_period[f'{ticker}_return'])
        joblib.dump(model, f'models/{ticker}_return_model.joblib')  # Save model to file
        models[ticker] = model
    return models, None, None  # No scaler and PCA in this simplified example

# Step 4: Mean-Variance Optimization
def mean_variance_optimization(expected_returns, cov_matrix, target_return):
    n = len(expected_returns)
    N = 100
    mus = [10**(5 * t / N - 1.0) for t in range(N)]

    S = matrix(cov_matrix)
    pbar = matrix(expected_returns, (n, 1), 'd')  # Ensure correct format for pbar

    G = -matrix(np.eye(n))
    h = matrix(0.0, (n, 1))
    A = matrix(1.0, (1, n))
    b = matrix(1.0)

    portfolios = [solvers.qp(mu*S, -pbar, G, h, A, b)['x'] for mu in mus]

    returns = np.array([np.dot(pbar.T, x)[0, 0] for x in portfolios])
    risks = np.array([np.sqrt(np.dot(x.T, S*x))[0, 0] for x in portfolios])

    return portfolios, returns, risks

# Step 5: Portfolio Evaluation
def evaluate_portfolio(portfolios, returns, risks):
    sharpe_ratios = np.divide(returns, risks, out=np.zeros_like(returns), where=(risks != 0))
    max_sharpe_idx = np.argmax(sharpe_ratios)

    optimal_portfolio = portfolios[max_sharpe_idx]
    max_sharpe_ratio = sharpe_ratios[max_sharpe_idx]

    return optimal_portfolio, max_sharpe_ratio

# Step 6: Example of Using Functions with Sample Test Data
tickers = ["AAPL", "MSFT"]
start_date = '2023-01-01'
end_date = '2023-01-10'

data = get_data(tickers, start_date, end_date)
features = feature_engineering(data)

train_period = features.loc[:'2023-01-06']
validation_period = features.loc['2023-01-07':'2023-01-08']
test_period = features.loc['2023-01-09':'2023-01-10']

models, _, _ = train_and_save_models(train_period, validation_period)

predicted_returns = np.array([model.predict(test_period.drop(columns=[f'{ticker}_return'])) for ticker, model in models.items()])
cov_matrix = np.cov(predicted_returns)

target_return = 0.02
portfolios, returns, risks = mean_variance_optimization(predicted_returns.mean(axis=1), cov_matrix, target_return)

optimal_portfolio, max_sharpe_ratio = evaluate_portfolio(portfolios, returns, risks)
print(f'Optimal Portfolio Weights: {optimal_portfolio}')
print(f'Max Sharpe Ratio: {max_sharpe_ratio}')

# Visualize the Efficient Frontier
plt.figure(figsize=(10, 6))
plt.scatter(risks, returns, c=np.divide(returns, risks, out=np.zeros_like(returns), where=(risks != 0)), marker='o', cmap='viridis')
plt.title('Efficient Frontier')
plt.xlabel('Risk (Standard Deviation)')
plt.ylabel('Return')
plt.colorbar(label='Sharpe Ratio')
plt.scatter(risks[np.argmax(returns / risks)], returns[np.argmax(returns / risks)], marker='*', color='r', s=200, label='Optimal Portfolio')
plt.legend()
plt.grid(True)
plt.show()
